# 02 · Preprocessing + RUL
**Owner:** Wijayagurusinghe · **Branch:** `data/02-preprocessing-rul`

Keep only cells that actually reach end-of-life, add the RUL label (`RUL = cycle_life − cycle`), do safe cleaning, and save `preprocessed.parquet`.


In [ ]:
# --- Colab setup (run first) ---
import os
if not os.path.exists('ev-battery-rul-prediction'):
    !git clone https://github.com/Chathumi-Senanayake/ev-battery-rul-prediction.git
%cd ev-battery-rul-prediction
!git pull -q
!pip install -q -r requirements.txt
import sys; sys.path.append('.')
from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/ev_battery_data'
os.makedirs(DATA_DIR, exist_ok=True)


In [ ]:
import pickle, pandas as pd
from src.preprocessing import (build_summary_table, add_rul, reached_eol,
    basic_clean, quality_report, print_report, EOL_AH)
bat_dict = pickle.load(open(os.path.join(DATA_DIR,'bat_dict.pkl'),'rb'))


### Drop cells that never reach EOL (≤ 0.88 Ah)


In [ ]:
eol = reached_eol(bat_dict)
bad = [c for c,ok in eol.items() if not ok]
print('cells not reaching EOL:', bad or 'none')
bat_dict = {c:d for c,d in bat_dict.items() if eol[c]}
print('cells kept:', len(bat_dict))


### Build table + RUL label + clean


In [ ]:
df = add_rul(build_summary_table(bat_dict))
clean = basic_clean(df, drop_first_cycle=True)
print('rows:', len(df), '->', len(clean), '| cells:', clean['cell'].nunique())
clean.to_parquet(os.path.join(DATA_DIR,'preprocessed.parquet'), index=False)
print_report(quality_report(clean))


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].hist(clean['RUL'],bins=50); ax[0].set_title('RUL'); ax[0].set_xlabel('cycles')
for c in list(clean['cell'].unique())[:15]:
    g=clean[clean['cell']==c]; ax[1].plot(g['cycle'],g['QD'],lw=.6)
ax[1].set_title('capacity fade'); ax[1].set_xlabel('cycle'); ax[1].set_ylabel('QD (Ah)')
plt.tight_layout(); plt.show()
